# DocStruct — FinanceBench + section-boundary run (Colab T4)

Upload this notebook, set **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Nothing else to do. Every cell is idempotent.

## It is meant to be run more than once

Free Colab reclaims sessions without warning, and this job is longer than one session.
Everything expensive lives on Drive: the corpora, the model weights, the detector cache,
the per-tool-per-document benchmark checkpoints, the PDF text spines and the section
checkpoints. **If the session dies, reopen the notebook and Run all again** — every stage
skips what is already done and picks up where it stopped. Re-running after a finished run
is cheap and changes nothing.

## What it produces

1. **FinanceBench leaderboard** — 84 born-digital SEC filings, 189 human-annotated
   evidence regions, `--relevance region` (mandatory on this corpus) with `--dump-scores`.
2. **Section-boundary agreement** — Pk / WindowDiff / straddle rate against 126 papers'
   publisher-authored JATS gold, including hybrid `docstruct`, which is too slow to run
   on a laptop.
3. **An offline sweep of `RELEVANCE_REGION_MIN_OVERLAP`**, which is still `# unvalidated`
   and which every region number in the paper currently rests on.

## 1. GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "NO GPU - Runtime > Change runtime type > T4 GPU, then Run all again."
print('GPU ok:', torch.cuda.get_device_name(0))

## 2. Drive — every expensive artefact lives here

`BENCH` is the single directory this notebook owns. Deleting it resets everything;
leaving it alone is what makes a re-run resume.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

BENCH    = '/content/drive/MyDrive/docstruct_bench'
CORPORA  = BENCH + '/corpora'        # fetched PDFs, so a dead session never refetches
CACHE    = BENCH + '/.bench_cache'   # benchmark checkpoints + detector proposals
DOTCACHE = BENCH + '/.cache'         # PDF text spines + section-scorer checkpoints
WEIGHTS  = BENCH + '/weights'
REPORTS  = BENCH + '/reports'
for d in (BENCH, CORPORA, CACHE, DOTCACHE, WEIGHTS, REPORTS):
    os.makedirs(d, exist_ok=True)
print('\n'.join((BENCH, CORPORA, CACHE, DOTCACHE, WEIGHTS, REPORTS)))

## 3. Repo and dependencies

In [ ]:
%cd /content
if not os.path.exists('/content/DocStruct/.git'):
    !git clone -q -b feat/paper-draft https://github.com/CandyButcher27/DocStruct
%cd /content/DocStruct
!git pull -q --ff-only 2>/dev/null || echo '(could not fast-forward; keeping what is here)'
!git log --oneline -1

In [ ]:
!pip install -q -e ".[all,benchmark-heavy]" unstructured-inference pyarrow    llama-index-core llama-index-embeddings-huggingface

# The install above churns Colab's preinstalled Pillow. Reinstalling cleanly fixes the
# files on disk, but not this kernel: Colab imports PIL at startup, so `PIL._typing` is
# already cached in sys.modules from the OLD version while the NEW ImageText.py is read
# fresh from disk. It then does `from ._typing import _Ink`, resolves against the stale
# cached module, and raises "cannot import name '_Ink' from 'PIL._typing'". The disk is
# consistent; sys.modules is not. Evicting PIL forces every submodule to reload from the
# new files, which is what a runtime restart would have achieved.
!pip install -q --force-reinstall --no-cache-dir pillow

import importlib
import sys
for _name in [m for m in list(sys.modules) if m == 'PIL' or m.startswith('PIL.')]:
    del sys.modules[_name]
importlib.invalidate_caches()

import PIL
from PIL import Image
Image.new('RGB', (4, 4))
print('pillow', PIL.__version__, 'ok')

# ImageText only exists from Pillow 12; it is the module that fails on a half-upgraded
# install, so it is worth checking when present -- but its absence is not an error.
try:
    from PIL import ImageText  # noqa: F401
    print('ImageText import ok')
except ImportError as e:
    print('ImageText not importable:', e, '- fine on Pillow < 12')

### Point the caches at Drive

`.cache/` holds the PDF text spines and the section-scorer checkpoints. Symlinking the
whole directory means a killed run resumes without any flag being passed.

In [ ]:
import os
if not os.path.islink('/content/DocStruct/.cache'):
    !rm -rf /content/DocStruct/.cache
    os.symlink(DOTCACHE, '/content/DocStruct/.cache')
print('.cache ->', os.path.realpath('/content/DocStruct/.cache'))

In [ ]:
# Which adapters actually imported. get_adapters() swallows import errors and drops
# anything whose available() is False, so a missing dependency would silently shrink the
# leaderboard instead of failing. Read the MISSING line before the long runs.
from docstruct.eval.adapters import get_adapters
WANT = ['docstruct', 'docstruct_geo', 'langchain', 'pymupdf4llm',
        'unstructured', 'llamaindex', 'llamaindex_semantic']
got = get_adapters(names=WANT, weights=None)
print('available:', sorted(got))
print('MISSING  :', sorted(set(WANT) - set(got)))
TOOLS = ','.join(n for n in WANT if n in got)
print('TOOLS    :', TOOLS)

## 4. Weights (cached on Drive)

In [ ]:
!mkdir -p weights
W = 'weights/yolov8m-doclaynet.pt'
if not os.path.exists(WEIGHTS + '/yolov8m-doclaynet.pt'):
    !wget -q --show-progress -O "{WEIGHTS}/yolov8m-doclaynet.pt" \
       https://huggingface.co/hantian/yolo-doclaynet/resolve/main/yolov8m-doclaynet.pt
!cp -n "{WEIGHTS}/yolov8m-doclaynet.pt" {W}
!ls -la weights/

# Confirm YOLO lands on the GPU. Nothing in docstruct/ sets a device; ultralytics and
# sentence-transformers auto-select CUDA. If this prints cpu, stop and say so.
from docstruct.model.detector import ModelDetector
_m = ModelDetector(weights=W)._ensure_model()
print('YOLO device:', next(_m.model.parameters()).device)

## 5. Corpora

Both are fetched **into Drive** and then copied to local disk. Drive is the source of
truth so a dead session never re-downloads; local disk is what the runs read, because
the benchmark reads every page repeatedly and Drive's FUSE mount is slow for that.

In [ ]:
# FinanceBench - 84 SEC filings, 189 human-annotated evidence regions, CC-BY-NC-4.0.
!mkdir -p data/financebench "{CORPORA}/financebench"
!cp -n "{CORPORA}/financebench/"*.pdf data/financebench/ 2>/dev/null || true
!python scripts/fetch_financebench.py
!cp -n data/financebench/*.pdf "{CORPORA}/financebench/" 2>/dev/null || true

import glob, json
print(len(glob.glob('data/financebench/*.pdf')), 'PDFs (expect 84)')
print(len(json.load(open('data/qa/financebench.json'))), 'gold rows (expect 189)')

In [ ]:
# PMC papers - 133 open-access papers across 7 journals, each with the publisher's JATS.
!mkdir -p data/pmc "{CORPORA}/pmc"
!cp -n "{CORPORA}/pmc/"* data/pmc/ 2>/dev/null || true
!python scripts/fetch_pmc.py --per-journal 20
!cp -n data/pmc/* "{CORPORA}/pmc/" 2>/dev/null || true
!python scripts/build_jats_gold.py

## 6. Smoke — two documents before anything long

Five failures in an earlier session were invisible to a green test suite and only
surfaced by running the real CLI. The suite passing is not evidence the run will start.

In [ ]:
import glob, json, os, shutil
!mkdir -p /content/smoke2
smoke = sorted(glob.glob('data/financebench/*.pdf'))[:2]
for p in smoke:
    shutil.copy(p, '/content/smoke2/')
names = {os.path.basename(p) for p in smoke}
gold = [g for g in json.load(open('data/qa/financebench.json')) if g['source_doc'] in names]
json.dump(gold, open('/content/smoke2_qa.json', 'w'))
print(len(gold), 'gold rows over', len(names), 'documents')

!python -m docstruct.cli benchmark \
   --pdfs-dir /content/smoke2 --qa /content/smoke2_qa.json \
   --weights {W} --tools docstruct_geo,langchain \
   --relevance region --dump-scores \
   --report-md /content/smoke.md --report-json /content/smoke.json

In [ ]:
# The dumped scores are the entire point of --dump-scores; assert they are present
# rather than discovering after the long run that the sweep has nothing to read.
d = json.load(open('/content/smoke.json'))
assert any('hyb_scores' in q for t in d['results'] for q in t['per_question']), \
    '--dump-scores produced no scores; the offline threshold sweep would have nothing to read'
print('smoke ok - dumped scores present')

## 7. FinanceBench — the long run

~15,000 pages, four times the OHR-Bench run, so **expect this to outlive one session**.
It checkpoints per tool per document into Drive; if the session dies, Run all again.

`--relevance region` is not a preference here. FinanceBench evidence is a page *region*
averaging 1,358 characters, and span-mode containment fails in proportion to how *small*
a tool chunks — 74% of regions are structurally uncontainable by unstructured's chunks
against 11% for ours. Span mode on this corpus would hand DocStruct a win it did not earn.

In [ ]:
!mkdir -p reports
!python -m docstruct.cli benchmark \
   --pdfs-dir data/financebench --qa data/qa/financebench.json \
   --weights {W} --tools {TOOLS} \
   --relevance region --dump-scores \
   --cache-dir "{CACHE}" \
   --report-md reports/fb_report_region.md --report-json reports/fb_results_region.json

In [ ]:
!cp -f reports/fb_report_region.md reports/fb_results_region.json "{REPORTS}/" 2>/dev/null || true
from IPython.display import Markdown, display
if os.path.exists('reports/fb_report_region.md'):
    display(Markdown(open('reports/fb_report_region.md').read()[:4000]))
else:
    print('no FinanceBench report yet - the run did not finish. Run all again; it resumes.')

## 8. Section-boundary agreement on the PMC papers

Does a chunker split where the document splits? Pk and WindowDiff against the
publisher's own JATS section boundaries — **lower is better**, 0.0 is perfect agreement.
Unlike section paths this is a real comparison: every chunker has boundaries.

The ceiling runs first. A gold boundary that cannot be found in the PDF's own text
cannot be scored against, and a Pk that quietly skipped a third of them would still look
like a result.

In [ ]:
!python scripts/section_reachability.py
!cp -f reports/section_reachability.json "{REPORTS}/" 2>/dev/null || true

In [ ]:
# Includes hybrid docstruct, which is why this wants a GPU: one figure-dense paper
# measured 475s geometry-only on a laptop CPU (notes.md Stage 18).
!python scripts/score_sections.py \
   --tools {TOOLS} --weights {W} --cache-dir "{CACHE}"
!cp -f reports/section_scores.md reports/section_scores.json "{REPORTS}/" 2>/dev/null || true
from IPython.display import Markdown, display
if os.path.exists('reports/section_scores.md'):
    display(Markdown(open('reports/section_scores.md').read()))

## 9. Sweep the region threshold — offline, seconds, no GPU

`RELEVANCE_REGION_MIN_OVERLAP = 0.7` is `# unvalidated`, and both the FinanceBench
leaderboard above and DocStruct's OHR-Bench region win rest on it. Because
`--dump-scores` recorded the continuous score behind every retrieved chunk, the
threshold is now a re-scoring rather than a re-run.

Read it for the **plateau**, not the peak: at 0.0 every chunk counts as relevant and MRR
is 1.0 by definition, so a metric that climbs as the threshold falls is not evidence for
a low threshold. The line that matters is whether the *ranking* moves.

In [ ]:
!python scripts/sweep_relevance_threshold.py \
   --results reports/fb_results_region.json \
   --out reports/fb_threshold_sweep.json
!cp -f reports/fb_threshold_sweep.json "{REPORTS}/" 2>/dev/null || true

## 10. Collect everything

In [ ]:
!cd reports && zip -q -r /content/docstruct_results.zip \
    fb_report_region.md fb_results_region.json fb_threshold_sweep.json \
    section_scores.md section_scores.json section_reachability.json 2>/dev/null || true
!cp -f /content/docstruct_results.zip "{BENCH}/" 2>/dev/null || true
!ls -la "{REPORTS}/"
try:
    from google.colab import files
    files.download('/content/docstruct_results.zip')
except Exception as e:
    print('browser download skipped:', e)
    print('the zip is on Drive at', BENCH)

---
## Back on the laptop

Commit the JSONs — they are the paper's evidence — and then:

1. **Read the threshold sweep first.** If the FinanceBench ranking moves with the
   threshold, no region number in the paper can be quoted without stating the value.
2. **Compare the model detector across corpora.** On OHR-Bench, `docstruct` vs
   `docstruct_geo` was +0.0012 (`span`, p=0.80) and +0.0090 (`region`, p=0.12) — neither
   significant. FinanceBench is the corpus built to show the detector's worth, because
   its tables are borderless and pdfplumber cannot see them: 122 detected tables against
   4 on `3M_2018_10K`. If it is null here too, that is a finding about the design.
3. **Section scores are new.** That metric found four defects in itself before producing
   a number; treat the first table as provisional.